<a href="https://colab.research.google.com/github/nishalahmedpk/timeseries-explainable/blob/test/GraphRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# 1. Open the files you dragged into Colab
icu_df = pd.read_csv('icu_items.csv')
lab_df = pd.read_csv('lab_items.csv')

# 2. Build the translation dictionary
item_mapping = {}

# We make everything lowercase so it matches PrimeKG exactly
for index, row in icu_df.iterrows():
    item_mapping[row['itemid']] = str(row['label']).lower()

for index, row in lab_df.iterrows():
    item_mapping[row['itemid']] = str(row['label']).lower()

# 3. Test it to see if it worked
test_mira_ids = [221906, 50912] # Norepinephrine and Creatinine
translated_words = [item_mapping[item_id] for item_id in test_mira_ids if item_id in item_mapping]

print("Success! The model's IDs translate to:", translated_words)

Success! The model's IDs translate to: ['norepinephrine']


In [2]:
import pandas as pd

# 1. Read the CSVs you exported from BigQuery
icu_df = pd.read_csv('icu_items.csv')
lab_df = pd.read_csv('lab_items.csv')

# 2. Combine them into one mapping dictionary
# This creates a dictionary like {221906: "norepinephrine", 50912: "creatinine"}
item_mapping = {}

for index, row in icu_df.iterrows():
    # Convert labels to lowercase so they match PrimeKG's format
    item_mapping[row['itemid']] = str(row['label']).lower()

for index, row in lab_df.iterrows():
    item_mapping[row['itemid']] = str(row['label']).lower()

# 3. Prep the input for the Agent
# Let's say MIRA flags these IDs for a specific patient:
mira_flagged_ids = [221906, 50912]

# You translate them using your BigQuery mapping:
patient_items_for_agent = [item_mapping[item_id] for item_id in mira_flagged_ids if item_id in item_mapping]

print("Ready for GraphRAG:", patient_items_for_agent)
# Output should be: Ready for GraphRAG: ['norepinephrine', 'creatinine']

Ready for GraphRAG: ['norepinephrine']


In [3]:
!pip install pandas networkx openai

In [4]:
import pandas as pd

# 1. Load the BigQuery files you uploaded
icu_df = pd.read_csv('icu_items.csv')
lab_df = pd.read_csv('lab_items.csv')

# 2. Build the dictionary mapping
item_mapping = {}

# Convert everything to lowercase to match PrimeKG
for index, row in icu_df.iterrows():
    item_mapping[row['itemid']] = str(row['label']).lower()

for index, row in lab_df.iterrows():
    item_mapping[row['itemid']] = str(row['label']).lower()

# 3. Test it! Let's mock what Nishal's model will send us:
mira_flagged_ids = [221906, 50912] # E.g., Norepinephrine and Creatinine

# Translate them:
patient_items = [item_mapping[item_id] for item_id in mira_flagged_ids if item_id in item_mapping]

print("Successfully translated IDs to words:", patient_items)

Successfully translated IDs to words: ['norepinephrine']


In [12]:
import pandas as pd
import networkx as nx

print("Loading PrimeKG into memory. This will take a moment...")

# Read the massive CSV
primekg_df = pd.read_csv('kg.csv')

# --- THE FIX ---
# Force every single node name in PrimeKG to be lowercase so it matches our inputs perfectly!
primekg_df['x_name'] = primekg_df['x_name'].astype(str).str.lower()
primekg_df['y_name'] = primekg_df['y_name'].astype(str).str.lower()

# Convert it into a NetworkX Directed Graph
G = nx.from_pandas_edgelist(
    primekg_df,
    source='x_name',
    target='y_name',
    edge_attr=['relation'],
    create_using=nx.DiGraph()
)

print(f"Success! Loaded PrimeKG with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Loading PrimeKG into memory. This will take a moment...


/tmp/ipykernel_937/2426862354.py:7: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  primekg_df = pd.read_csv('kg.csv')


Success! Loaded PrimeKG with 126010 nodes and 5294894 edges.


In [9]:
!pip install -U google-generativeai

In [17]:
import time
import networkx as nx
import google.generativeai as genai
from google.colab import userdata

# 1. SECURELY LOAD YOUR API KEY
gemini_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=gemini_key)

# 2. LOAD THE MODEL (Automatically selects the best available Flash model)
available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
target_model = 'models/gemini-2.5-flash' if 'models/gemini-2.5-flash' in available_models else available_models[0]
llm = genai.GenerativeModel(target_model)

def tier1_gp_agent(items):
    """Tier 1: Uses Gemini, and automatically waits if the server is busy."""
    prompt = f"""
    You are a clinical triage AI. The patient is exhibiting abnormalities related to these items: {items}.
    Identify the single most likely broad clinical context or disease category (e.g., 'sepsis', 'cardiac failure', 'acute kidney injury').
    Respond with ONLY the disease name in lowercase. Do not include any other text.
    """

    # Try 3 times. If we hit the Free Tier limit, pause for 60 seconds and try again.
    for attempt in range(3):
        try:
            response = llm.generate_content(prompt)
            return response.text.strip().lower()
        except Exception as e:
            if '429' in str(e) or 'Quota' in str(e) or 'TooManyRequests' in str(e):
                print(f"⚠️ API busy. Automatically waiting 60 seconds... (Attempt {attempt + 1}/3)")
                time.sleep(60) # Pauses the code for 60 seconds
            else:
                raise e # If it's a completely different error, crash normally

    return "unknown_context"

def tier2_specialized_agent(graph, items, target_disease):
    """Tier 2: Traverses PrimeKG using ONLY clinically relevant pathways."""
    subgraph_data = []

    # We only want real medical relationships, not random side effects
    valid_relations = [
        'indication',                 # Drug treats disease
        'off-label use',              # Drug treats disease (unofficial)
        'disease_phenotype_positive', # Disease causes symptom/lab abnormality
        'disease_disease',            # Disease is linked to another disease
        'phenotype_phenotype'         # Symptom causes symptom
    ]

    # Build a temporary, filtered graph that only contains these valid highways
    valid_edges = [
        (u, v) for u, v, data in graph.edges(data=True)
        if data['relation'] in valid_relations
    ]
    clinical_graph = graph.edge_subgraph(valid_edges)

    for item in items:
        try:
            # Search the CLEAN clinical graph
            path = nx.shortest_path(clinical_graph, source=item, target=target_disease)

            # Extract the connections for the final dictionary
            for i in range(len(path) - 1):
                source_node = path[i]
                target_node = path[i+1]
                relation = clinical_graph[source_node][target_node]['relation']

                subgraph_data.append({
                    "source": source_node,
                    "relation": relation,
                    "target": target_node
                })
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            print(f"Warning: No clean clinical path found between '{item}' and '{target_disease}'.")

    return subgraph_data

# --- RUN THE PIPELINE ---
print("--- Running GraphRAG Pipeline ---")

# Execute Phase 1: AI Diagnosis
disease_context = tier1_gp_agent(patient_items)
print(f"Tier 1 GP Agent Diagnosis: {disease_context}")

# Execute Phase 2: Biological Verification
medical_subgraph = tier2_specialized_agent(G, patient_items, disease_context)

# Package for MoE Router
final_output = {
    "identified_context": disease_context,
    "input_metadata": patient_items,
    "subgraph": medical_subgraph
}

print("\nFinal Output for MoE Router:")
print(final_output)

--- Running GraphRAG Pipeline ---


⚠️ API busy. Automatically waiting 60 seconds... (Attempt 1/3)


⚠️ API busy. Automatically waiting 60 seconds... (Attempt 2/3)


⚠️ API busy. Automatically waiting 60 seconds... (Attempt 3/3)
Tier 1 GP Agent Diagnosis: unknown_context

Final Output for MoE Router:
{'identified_context': 'unknown_context', 'input_metadata': ['norepinephrine'], 'subgraph': []}


In [11]:
# 1. Check if 'norepinephrine' exists exactly as spelled
if 'norepinephrine' in G.nodes():
    print("✅ 'norepinephrine' exists in PrimeKG!")
else:
    print("❌ 'norepinephrine' NOT FOUND. Let's search for similar names:")
    # Search for anything containing 'norepine'
    similar_meds = [n for n in G.nodes() if isinstance(n, str) and 'norepine' in n.lower()]
    print("Similar meds in PrimeKG:", similar_meds[:5])

print("\n-----------------------------------\n")

# 2. Search for how PrimeKG spells 'shock'
print("🔍 Searching for nodes containing the word 'shock':")
shock_nodes = [n for n in G.nodes() if isinstance(n, str) and 'shock' in n.lower()]

# Print the first 10 matches
for node in shock_nodes[:10]:
    print(f"- {node}")

❌ 'norepinephrine' NOT FOUND. Let's search for similar names:
Similar meds in PrimeKG: ['Norepinephrine', 'Elevated urinary norepinephrine']

-----------------------------------

🔍 Searching for nodes containing the word 'shock':
- streptococcal toxic-shock syndrome
- toxic shock syndrome
- staphylococcal toxic-shock syndrome
- Anaphylactic shock
- Cardiogenic shock
- Shock
- Hypovolemic shock
- Distributive shock
- Obstructive shock
